# NB2 — RepViT Full Ablation Battery — TopoDistil++

**Pipeline stage:** online (GPU), the primary backbone's full battery (Section 4.3, rows 1–13).

**Input:** this notebook's *File → Add Input* must include NB1's output
(`01_data_topology_prep`), which provides `A_gauss` maps, persistence diagrams,
per-patch entropy, and the fixed split — all precomputed offline so no PH computation
happens in this notebook.

**What this notebook does:**
1. Load the NB1 checkpoint and build a topology-aware `Dataset`/`DataLoader`.
2. Implement the model components from Section 3.4–3.9: the learned residual projector
   `f_θ`, the topology-guided gate (Form A / Form B), the TopKD-style embedding baseline,
   and a generic (non-topological) Attention Transfer baseline.
3. Run all 13 settings from Section 4.3's table on RepViT — 25 total runs (rows 1, 3, 4,
   10, 11, 13 at 3 seeds each; the remaining 7 rows at 1 seed) — via a single
   config-driven, resumable loop. **Update 1:** row 4 (`row4_lattn_only_no_gate`,
   distillation-only, no gate) was promoted to a 3-seed core row after its single-seed
   result (0.9358) beat full LIFE on RepViT; under the old training loop the effect did
   not survive 3 seeds, and is being re-tested under the stabilized loop below.
   **Update 2:** row 13 (`row13_oracle`) is also promoted to 3 seeds, since its n=1
   ceiling had come out *below* baseline -- structurally implausible, and now testable
   properly. **Update 3 (training-stability fix, Section 1 item 8):** the training loop
   now uses discriminative learning rates, a warmup+cosine schedule, and best-val-AUC
   checkpoint selection, targeting the seed-to-seed instability that was the likely
   dominant noise source behind both of the above.
4. Save per-run metrics, gating-magnitude logs, and weights (core rows only) as a
   checkpoint bundle for NB4 (analysis) and, for the cross-check settings, for NB3.

> A few implementation details in the paper are underspecified for an engineering build
> (e.g. exactly how H₀/H₁ gate channels combine, what "generic spatial signal" means for
> the Attention Transfer baseline). See **Section 1 — Implementation assumptions** below
> for every such decision, made explicit and adjustable via `CONFIG`.


## 0. Setup

In [ ]:

!pip install -q timm thop


In [ ]:

import os, json, time, pickle, random, math, copy
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

import timm

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 1. Implementation assumptions

The paper specifies the method precisely at the level of loss functions and gate
formulas, but a few engineering choices aren't pinned down. Made explicit here so
they're easy to challenge/change in one place:

1. **Privileged-information framing for "removed at inference."** The gate multiplies
   `F` by a function of `A_proj`, which only exists pre-computed for the *train* split
   (Section 4.1: PH extraction is offline, on train only). We treat topology as
   privileged information available at train time only: during training the gate is
   always applied using the real `A_proj`; standard validation/test evaluation runs with
   the gate **bypassed** (`F' = F`, no topology needed); the **oracle** setting
   (row 13) is the only one that computes `A_proj` on the fly for val/test patches too,
   at the stated cost of test-time PH.
2. **H₀/H₁ gate combination.** `A_proj` has two channels (H₀, H₁ maps). We combine them
   into a single scalar gate map via a learned `1×1` conv (`gate_combine`) rather than
   averaging — lets the network weight the two dimensions instead of assuming equal
   importance a priori.
3. **`A_student` / `L_attn`.** A `1×1` conv head (`TopoAlignHead`) projects the
   intermediate feature map `F` down to a 2-channel spatial map, resized to match
   `A_proj`'s resolution; `L_attn` is the MSE between the two, computed at `F`'s spatial
   resolution (cheaper, and the loss target is nearest/bilinear-resized to match).
4. **Attention Transfer baseline (row 2).** No teacher network exists in this
   single-network design, so "generic spatial signal" is implemented as a Sobel
   gradient-magnitude map of the input image — same `L_attn` machinery, non-topological
   target, no gating. This isolates "any spatial supervision helps" from
   "topology specifically helps."
5. **Shuffled control (row 11).** Implemented as a per-sample random pixel permutation
   of `A_proj` — preserves the value histogram (density statistics) while destroying
   spatial correctness, matching the paper's stated purpose for this ablation.
6. **Primary gate form for rows 11–13.** These ablations are defined relative to the
   "full" method; we apply them on top of **Form B** (the form also cross-checked on
   MobileNetV4 in NB3), consistent with row 10 being the second core-credibility row.
7. **Injection layer.** A single fixed mid-level layer, chosen once via
   `pick_injection_layer()` below and held constant across all 25 runs, per Section 3.5.
   (Not yet swept against alternative depths -- flagged as a follow-up once the
   training-stability fix below is validated.)
8. **Training stability fix.** The original loop fine-tuned the entire pretrained
   backbone at a flat `lr=1e-3` with no schedule, for a fixed 12 epochs, and reported
   whatever the model happened to be at epoch 12 -- no best-checkpoint selection. This
   is a plausible dominant source of the seed-to-seed AUC variance (~0.01-0.03) that
   was larger than any effect under test, and is consistent with the full battery's
   baseline AUC shifting substantially (0.9314 -> 0.9217) between two nominally-identical
   from-scratch reruns. The fix: (a) discriminative learning rates -- `LR_BACKBONE`
   (lower, for the pretrained backbone) vs `LR` (higher, for freshly-initialized
   gate/residual-projector/align-head/topkd modules); (b) a 1-epoch linear warmup
   followed by cosine decay to `MIN_LR_FRAC` of base LR, applied per optimizer step;
   (c) per-epoch validation AUC tracking, with the best-val checkpoint restored before
   every reported evaluation (standard val/test *and* oracle), replacing "whatever
   epoch 12 was" with "the checkpoint that was actually best on held-out data." See
   Section 10's `train_one_run` for the implementation.


## 2. Config

In [ ]:

def find_nb1_dir(root="/kaggle/input/notebooks/claudeisnotclaude/data-setup"):
    root = Path(root)
    for f in root.rglob("splits.json"):
        return f.parent
    return None

nb1_dir = find_nb1_dir()
if nb1_dir is None:
    print("Could not find NB1's output. Add it via File > Add Input > Notebook Output,")
    print("then re-run this cell, or set CONFIG['NB1_DIR'] manually below.")
else:
    print("Found NB1 checkpoint at:", nb1_dir)


In [ ]:

CONFIG = {
    "NB1_DIR": str(nb1_dir) if nb1_dir else "/kaggle/input/<nb1-slug>/topology_checkpoint",
    "PCAM_DIR": "/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon",

    "BACKBONE": "repvit_m1.dist_in1k",
    "INJECTION_LAYER": None,        # filled in by pick_injection_layer() below
    "IMG_SIZE": 96,

    "BATCH_SIZE": 32,
    "LR": 1e-3,                     # LR for freshly-initialized modules (gate, residual
                                     # projector, align head, topkd baseline)
    "LR_BACKBONE": 1e-4,            # LR for the pretrained backbone -- 10x lower, since it
                                     # needs gentler treatment than randomly-init modules
                                     # (Section 1, item 8: discriminative-LR fix)
    "WARMUP_EPOCHS": 1,             # linear warmup, then cosine decay for the rest of training
    "MIN_LR_FRAC": 0.05,            # cosine decays down to 5% of each group's base LR
    "EPOCHS": 12,
    "LAMBDA": 0.5,                  # weight on L_attn, per Section 3.7 (fixed except row 5)

    "VAL_SUBSET": 5000,             # patches used for standard val/test eval
    "TEST_SUBSET": 5000,
    "ORACLE_SUBSET": 1000,          # oracle eval needs on-the-fly PH -> keep small

    "SEEDS_CORE": [0, 1, 2],        # rows 1, 3, 4, 10, 11, 13 (row 4 promoted earlier; row 13
                                     # oracle promoted now too -- n=1 was the reason its ceiling
                                     # looked inconclusive/inverted, see Section 11 note)
    "SEED_SINGLE": 0,               # rows 2, 5, 6, 7, 8, 9, 12, 13

    "OUT_DIR": "/kaggle/working/repvit_results",
    "SAVE_WEIGHTS_FOR": {"row1", "row3", "row4", "row10", "row11", "row13"},  # core rows + oracle (row4 added)
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

CONFIG


## 2b. Resume from a previous session's output (if attached as input)

`/kaggle/working/` always starts empty in a new session -- that's normal Kaggle
behavior. If a previous version of *this* notebook's output was attached via
*File > Add Input*, it lands under `/kaggle/input/...`, which is read-only and which
the resume check in Section 12 (`manifest_path = f'{CONFIG["OUT_DIR"]}/run_manifest.json'`)
never looked at -- it only ever checked `CONFIG["OUT_DIR"]` itself. That's why the last
run appeared to start over from row 1 despite `run_manifest.json` showing 22/25 runs
`done`: the manifest genuinely wasn't found, not because the run wasn't there, but
because it was never looked for in the right place.

This cell finds a previous `repvit_results/run_manifest.json` under `/kaggle/input/` (the
same pattern `find_nb1_dir()` above already uses for NB1, just pointed at *this*
notebook's own output signature) and copies everything -- manifest, metrics, weights --
into `CONFIG["OUT_DIR"]` so Section 12's resume check finds it. Safe to run whether or
not a previous output is attached: if none is found, it prints a message and changes
nothing, and the battery just starts fresh.

In [ ]:
import shutil

def find_previous_nb2_output(out_dir_name="repvit_results", root="/kaggle/input"):
    """Search /kaggle/input for a previous run of *this* notebook (identified by its own
    manifest file, run_manifest.json, inside a directory named like CONFIG["OUT_DIR"]'s
    basename) and return its path, or None if nothing is attached."""
    root = Path(root)
    if not root.exists():
        return None
    for f in root.rglob("run_manifest.json"):
        if f.parent.name == out_dir_name or f.parent.parent.name == out_dir_name:
            return f.parent
    return None

prev_dir = find_previous_nb2_output(out_dir_name=Path(CONFIG["OUT_DIR"]).name)

if prev_dir is None:
    print("No previous NB2 output found under /kaggle/input -- starting fresh "
          "(this is expected on a first run, or if you haven't attached a previous "
          "version's output yet via File > Add Input).")
else:
    prev_manifest = json.load(open(prev_dir / "run_manifest.json"))
    n_done = sum(1 for v in prev_manifest.values() if v.get("status") == "done")
    print(f"Found previous NB2 output at {prev_dir} ({n_done} runs already done).")
    print(f"Copying into {CONFIG['OUT_DIR']} so the resume check in Section 12 finds it...")
    shutil.copytree(prev_dir, CONFIG["OUT_DIR"], dirs_exist_ok=True)

    # Verify the copy actually landed where Section 12 will look for it -- this is a
    # read-back check, not just a "did copytree raise" check, since a silent path
    # mismatch here would reproduce exactly the bug this cell is fixing.
    check_path = Path(CONFIG["OUT_DIR"]) / "run_manifest.json"
    assert check_path.exists(), (
        f"Copy appeared to succeed but {check_path} still doesn't exist -- "
        f"CONFIG['OUT_DIR'] may not match what Section 12 uses. Stop and check before "
        f"running the battery, or it will silently retrain everything again."
    )
    copied_manifest = json.load(open(check_path))
    n_copied_done = sum(1 for v in copied_manifest.values() if v.get("status") == "done")
    n_weight_files = len(list(Path(CONFIG["OUT_DIR"]).glob("*_weights.pt")))
    print(f"Verified: {n_copied_done} done runs and {n_weight_files} weight files now "
          f"present in {CONFIG['OUT_DIR']}. Section 12 will resume from here.")


## 3. Load the NB1 checkpoint

In [ ]:

with open(f'{CONFIG["NB1_DIR"]}/splits.json') as f:
    splits = json.load(f)
with open(f'{CONFIG["NB1_DIR"]}/config_used.json') as f:
    nb1_config = json.load(f)

patch_meta = pd.read_csv(f'{CONFIG["NB1_DIR"]}/patch_meta.csv').set_index("patch_idx")

with open(f'{CONFIG["NB1_DIR"]}/persistence_diagrams.pkl', "rb") as f:
    diagrams_store = pickle.load(f)

h5_maps_path = f'{CONFIG["NB1_DIR"]}/A_gauss_maps.h5'
with h5py.File(h5_maps_path, "r") as f:
    map_patch_order = np.array(f["patch_idx"])
patch_to_row = {int(p): i for i, p in enumerate(map_patch_order)}

train_idx = np.array(splits["train_subsample_idx"])
train_labels = np.array(splits["train_subsample_labels"])
H_THRESH = float(np.median(patch_meta.loc[train_idx, ["entropy_h0", "entropy_h1"]].sum(axis=1)))
CONFIG["H_THRESH"] = H_THRESH

print(f"Train subsample: {len(train_idx)} patches")
print(f"H_thresh (median summed entropy): {H_THRESH:.3f}")

# PCam files (needed again here for images; same discovery as NB1)
from pathlib import Path

def find_pcam_files(root):
    root = Path(root)

    found = {}

    # Image HDF5 files in this Kaggle dataset
    image_names = {
        "train_x": "training_split.h5",
        "valid_x": "validation_split.h5",
        "test_x": "test_split.h5",
    }

    # Label HDF5 files
    label_names = {
        "train_y": "camelyonpatch_level_2_split_train_y.h5",
        "valid_y": "camelyonpatch_level_2_split_valid_y.h5",
        "test_y": "camelyonpatch_level_2_split_test_y.h5",
    }

    all_h5 = list(root.rglob("*.h5"))

    for key, filename in {**image_names, **label_names}.items():
        matches = [f for f in all_h5 if f.name == filename]
        if matches:
            found[key] = matches[0]

    return found


pcam_files = find_pcam_files(CONFIG["PCAM_DIR"])

required = [
    "train_x", "train_y",
    "valid_x", "valid_y",
    "test_x", "test_y"
]

missing = [k for k in required if k not in pcam_files]

if missing:
    print("Could not find:", missing)
    print("\nH5 files found:")
    for f in sorted(Path(CONFIG["PCAM_DIR"]).rglob("*.h5")):
        print(" ", f)
else:
    print("Found all PCam files:")
    for k, v in pcam_files.items():
        print(f"  {k}: {v}")

## 4. Dataset

Loads image + `A_gauss` map + entropy for a given patch index. Flip/90°-rotation
augmentation is applied to the image and, in **aligned** mode (default, Section 3.8),
the identical transform is applied to the map array directly (equivalent to
transforming the critical-point coordinates, since the map is already rasterized).
**Unaligned** mode (row 12) applies an independently-sampled transform to the map,
breaking the alignment on purpose.

> **Fix:** the row-11 shuffled control must permute **`A_proj`** (Section 1,
> assumption 5) -- the post-residual, post-homology-selection map actually used for
> gating and `L_attn`, not the raw fixed `A_gauss` prior. `A_proj = A_gauss +
> f_theta(A_gauss)`, and `f_theta` is a spatial conv net, so permuting `A_gauss`
> pixels *before* it runs does not equal permuting `A_proj` pixels *after* it runs --
> the conv mixes each pixel with now-scrambled neighbors, so neither the destroyed-
> spatial-structure nor the preserved-value-histogram property the control is
> supposed to have was guaranteed. The shuffle now happens in `TopoDistilModel`,
> after `build_A_proj`, on the exact tensor used downstream (see Sections 5-6).


In [ ]:

def augment_pair(img, amap, mode, rng):
    '''img: (H,W,3) uint8-like float array. amap: (2,H,W) float array.'''
    k = rng.integers(0, 4)          # 90-degree rotations
    flip = rng.integers(0, 2)       # horizontal flip

    img_t = np.rot90(img, k, axes=(0, 1))
    if flip:
        img_t = np.flip(img_t, axis=1)

    if mode == "aligned":
        amap_t = np.rot90(amap, k, axes=(1, 2))
        if flip:
            amap_t = np.flip(amap_t, axis=2)
    else:  # unaligned: independent random transform on the map
        k2 = rng.integers(0, 4)
        flip2 = rng.integers(0, 2)
        amap_t = np.rot90(amap, k2, axes=(1, 2))
        if flip2:
            amap_t = np.flip(amap_t, axis=2)

    return np.ascontiguousarray(img_t), np.ascontiguousarray(amap_t)


class PCamTopoDataset(Dataset):
    # NOTE: no shuffle_map here anymore -- the row-11 shuffled control permutes
    # A_proj (post-residual, post-homology-selection), which only exists inside the
    # model forward pass. See TopoDistilModel / build_A_proj below (Section 5-6 fix).
    def __init__(self, indices, split, labels=None, augment=True, aug_mode="aligned",
                 seed=0):
        self.indices = np.asarray(indices)
        self.split = split
        self.labels = labels
        self.augment = augment
        self.aug_mode = aug_mode
        self.rng = np.random.default_rng(seed)
        self._h5 = None
        self._h5_maps = None  # lazy handle for A_gauss_maps.h5 (was reopened per __getitem__)

    def _images(self):
        if self._h5 is None:
            self._h5 = h5py.File(pcam_files[f"{self.split}_x"], "r")
            self._x_key = list(self._h5.keys())[0]
        return self._h5[self._x_key]

    def _maps(self):
        if self._h5_maps is None:
            self._h5_maps = h5py.File(h5_maps_path, "r")
        return self._h5_maps["A_gauss"]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = np.array(self._images()[idx]).astype(np.float32) / 255.0

        if self.split == "train" and idx in patch_to_row:
            row = patch_to_row[idx]
            amap = np.array(self._maps()[row]).astype(np.float32)
            # entropy_h0/entropy_h1 kept SEPARATE (not summed) -- summed only where the
            # gate's gamma() genuinely needs a single scalar (see TopoGate.forward).
            entropy = patch_meta.loc[idx, ["entropy_h0", "entropy_h1"]].to_numpy(dtype=np.float32)
        else:
            amap = np.zeros((2, CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), dtype=np.float32)
            entropy = np.zeros(2, dtype=np.float32)

        if self.augment:
            img, amap = augment_pair(img, amap, self.aug_mode, self.rng)

        label = int(self.labels[i]) if self.labels is not None else -1
        img_t = torch.from_numpy(img.transpose(2, 0, 1).copy()).float()
        amap_t = torch.from_numpy(amap.copy()).float()
        return img_t, amap_t, entropy, label, idx


## 5. Model components (Sections 3.4–3.6)

In [ ]:

class ResidualProjector(nn.Module):
    '''f_theta: learned residual on top of the fixed Gaussian prior (Section 3.4, Step 2).'''
    def __init__(self, channels=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, 3, padding=1), nn.BatchNorm2d(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 3, padding=1),
        )

    def forward(self, a_gauss):
        return self.net(a_gauss)


def build_A_proj(a_gauss, residual_projector, use_residual, homology_mode):
    '''A_proj = A_gauss + f_theta(A_gauss), with H0/H1 channel selection.'''
    if use_residual:
        a_proj = a_gauss + residual_projector(a_gauss)
    else:
        a_proj = a_gauss  # row 6: fixed-A_gauss-only ablation

    if homology_mode == "h0":
        a_proj = torch.stack([a_proj[:, 0], torch.zeros_like(a_proj[:, 0])], dim=1)
    elif homology_mode == "h1":
        a_proj = torch.stack([torch.zeros_like(a_proj[:, 1]), a_proj[:, 1]], dim=1)
    return a_proj


class TopoGate(nn.Module):
    '''Section 3.5. form='A' -> multiplicative dual gate. form='B' -> residual gate.'''
    def __init__(self, form="A"):
        super().__init__()
        self.form = form
        self.gate_combine = nn.Conv2d(2, 1, kernel_size=1)   # H0/H1 -> single gate map
        self.beta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.last_gate_mean = None  # for the gating-magnitude / collapse check (Section 4.5)

    def forward(self, feat, a_proj, entropy, h_thresh):
        a_resized = F_.interpolate(a_proj, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        combined = self.gate_combine(a_resized)
        gate = torch.sigmoid(self.beta * combined + self.alpha)
        # entropy is (B,2) = [entropy_h0, entropy_h1] kept separate for the TopKD baseline;
        # gamma still gates on the combined (summed) entropy against H_THRESH, per Section 3.5.
        entropy_sum = entropy.sum(dim=1) if entropy.dim() > 1 else entropy
        gamma = torch.sigmoid(entropy_sum - h_thresh).view(-1, 1, 1, 1)
        self.last_gate_mean = float((gate * gamma).mean().detach().cpu())

        if self.form == "A":
            return feat * gate * gamma
        else:  # form B
            return feat + feat * gate * gamma


class TopoAlignHead(nn.Module):
    '''Projects F down to a 2-channel spatial map for L_attn (Section 3.7).'''
    def __init__(self, in_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, 2, kernel_size=1)

    def forward(self, feat):
        return self.proj(feat)


class TopKDEmbeddingBaseline(nn.Module):
    '''Section 3.6: persistence-summary-stats -> global embedding, aligned to the
    student's pooled penultimate embedding via MSE. No spatial component, no gating.'''
    def __init__(self, embed_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(6, 32), nn.ReLU(inplace=True),
            nn.Linear(32, embed_dim),
        )

    def summary_stats(self, amap, entropy):
        # [total_persistence_h0, total_persistence_h1, entropy_h0, entropy_h1, n_h0, n_h1]
        # amap channels are already persistence-weighted Gaussian maps; total_persistence
        # is approximated as the channel sum (a monotonic proxy for sum p_i used offline).
        tot_h0 = amap[:, 0].sum(dim=(1, 2))
        tot_h1 = amap[:, 1].sum(dim=(1, 2))
        # entropy is (B,2) = [entropy_h0, entropy_h1] -- genuinely separate values now,
        # not the same combined scalar duplicated into both slots.
        ent_h0 = entropy[:, 0]
        ent_h1 = entropy[:, 1]
        n_h0 = (amap[:, 0] > 0).float().sum(dim=(1, 2))
        n_h1 = (amap[:, 1] > 0).float().sum(dim=(1, 2))
        stats = torch.stack([tot_h0, tot_h1, ent_h0, ent_h1, n_h0, n_h1], dim=1)
        return stats

    def forward(self, amap, entropy):
        stats = self.summary_stats(amap, entropy)
        return self.encoder(stats)


def sobel_map(img):
    '''Generic (non-topological) spatial saliency target for the Attention Transfer
    baseline (row 2). img: (B,3,H,W) in [0,1].'''
    gray = img.mean(dim=1, keepdim=True)
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=img.dtype, device=img.device).view(1, 1, 3, 3)
    ky = kx.transpose(2, 3)
    gx = F_.conv2d(gray, kx, padding=1)
    gy = F_.conv2d(gray, ky, padding=1)
    mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)
    mag = mag / (mag.amax(dim=(2, 3), keepdim=True) + 1e-8)
    return mag.repeat(1, 2, 1, 1)  # 2 channels to match A_proj's shape


def shuffle_spatial_per_sample(a_proj):
    '''Row 11 shuffled control (Section 1, assumption 5): per-sample random pixel
    permutation of A_proj, applied AFTER the residual projector and homology-mode
    selection so it operates on the exact map used for gating/L_attn, not on the raw
    fixed A_gauss prior. Same permutation is used across both channels of a given
    sample so H0/H1 stay spatially co-registered with each other (just not with the
    image); this exactly preserves each sample's per-channel value histogram while
    fully destroying spatial correctness -- matching the control's stated purpose.'''
    b, c, h, w = a_proj.shape
    flat = a_proj.reshape(b, c, h * w)
    out = torch.empty_like(flat)
    for i in range(b):
        perm = torch.randperm(h * w, device=a_proj.device)
        out[i] = flat[i][:, perm]
    return out.reshape(b, c, h, w)


## 6. Backbone wrapper

Injects the gate at a single fixed mid-level layer via a forward hook, per Section 3.5.
Run `pick_injection_layer()` once, confirm the chosen layer name looks like a
mid-network stage, and keep `CONFIG["INJECTION_LAYER"]` fixed for every run afterward.


In [ ]:

def pick_injection_layer(backbone_name=CONFIG["BACKBONE"]):
    model = timm.create_model(backbone_name, pretrained=True, num_classes=2)
    names = [n for n, _ in model.named_modules() if n]
    print(f"{len(names)} named modules in {backbone_name}. First/last 15 shown; "
          "pick one roughly in the middle of the stage hierarchy:")
    print(names[:15]); print("..."); print(names[-15:])
    return model, names

_probe_model, _all_names = pick_injection_layer()


In [ ]:

# Fill this in after inspecting the printed module names above -- pick a layer name
# that sits at roughly the midpoint of the backbone's stages. Example placeholder:
CONFIG["INJECTION_LAYER"] = CONFIG["INJECTION_LAYER"] or _all_names[len(_all_names) // 2]
print("Using injection layer:", CONFIG["INJECTION_LAYER"])
del _probe_model


In [ ]:

class TopoDistilModel(nn.Module):
    def __init__(self, backbone_name, gate_active, gate_form, use_L_attn,
                 use_residual, homology_mode, baseline_type, shuffled_control=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=2)
        self.gate_active = gate_active
        self.use_L_attn = use_L_attn
        self.use_residual = use_residual
        self.homology_mode = homology_mode
        self.baseline_type = baseline_type   # None | "topkd" | "attention_transfer"
        self.shuffled_control = shuffled_control   # row 11: permute A_proj, not A_gauss

        self.residual_projector = ResidualProjector() if use_residual else None
        self.gate = TopoGate(form=gate_form) if gate_active else None

        self.align_head = None      # lazy-init once we know F's channel count
        self.topkd_baseline = None
        self.use_topology = True    # flipped off for standard eval, on for oracle eval

        self._captured_feat = {}
        self._current_a_proj = None  # cached each forward pass so compute_losses can
                                      # reuse the exact (possibly shuffled) map the gate saw
        self._hook_handle = self.backbone.get_submodule(CONFIG["INJECTION_LAYER"]) \
                                          .register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self._captured_feat["F"] = out
        if not self.gate_active or not self.use_topology:
            return out
        a_proj = build_A_proj(self._current_a_gauss, self.residual_projector,
                               self.use_residual, self.homology_mode)
        if self.shuffled_control:
            # Fix: shuffle A_proj itself (post-residual, post-homology-selection) --
            # NOT the raw A_gauss prior -- so the control actually gets the value-
            # histogram-preserving, spatially-scrambled map described in Section 1,
            # assumption 5. Cached below so compute_losses uses this exact same
            # shuffled tensor as the L_attn target, instead of an independently
            # (differently) shuffled one.
            a_proj = shuffle_spatial_per_sample(a_proj)
        self._current_a_proj = a_proj
        out = self.gate(out, a_proj, self._current_entropy, CONFIG["H_THRESH"])
        return out

    def forward(self, img, a_gauss, entropy):
        self._current_a_gauss = a_gauss
        self._current_entropy = entropy
        self._current_a_proj = None
        logits = self.backbone(img)
        feat = self._captured_feat["F"]

        if self.align_head is None and self.use_L_attn:
            self.align_head = TopoAlignHead(feat.shape[1]).to(img.device)
        if self.topkd_baseline is None and self.baseline_type == "topkd":
            self.topkd_baseline = TopKDEmbeddingBaseline(feat.shape[1]).to(img.device)

        return logits, feat


## 7. Loss computation per setting

In [ ]:

def compute_losses(model, logits, feat, img, a_gauss, entropy, labels, lam, cfg):
    ce = F_.cross_entropy(logits, labels)
    total = ce
    l_attn_value = None

    if cfg["use_L_attn"] and lam > 0:
        a_student = model.align_head(feat)
        if cfg["baseline_type"] == "attention_transfer":
            target = sobel_map(img)
        elif model._current_a_proj is not None:
            # Fix: whenever the gate was active this forward pass, _hook() already
            # ran build_A_proj() once (through residual_projector, which contains a
            # BatchNorm2d) to get the gate's input. Reuse that cached value here
            # instead of calling build_A_proj() again -- recomputing it would run
            # residual_projector a second time on the identical input, giving it a
            # second BatchNorm running-stats EMA update per optimizer step (and,
            # for the shuffled control, an independently-shuffled map that would no
            # longer match what the gate saw). This covers rows 9/10/11/12/13.
            target = model._current_a_proj
        else:
            # Only reached when gate_active=False (e.g. row4_lattn_only_no_gate),
            # so build_A_proj hasn't been called yet this forward pass.
            target = build_A_proj(a_gauss, model.residual_projector, cfg["use_residual"],
                                   cfg["homology_mode"])
        target_resized = F_.interpolate(target, size=a_student.shape[-2:], mode="bilinear",
                                         align_corners=False)
        l_attn = F_.mse_loss(a_student, target_resized)
        total = total + lam * l_attn
        l_attn_value = float(l_attn.detach().cpu())

    if cfg["baseline_type"] == "topkd" and lam > 0:
        pooled = feat.mean(dim=(2, 3))
        topo_embed = model.topkd_baseline(a_gauss, entropy)
        # Deliberately NOT detached: there's no separate teacher network in this
        # single-network design (Section 1, assumption 4), so topkd_baseline's own
        # weights only ever receive gradient through this loss. Detaching would freeze
        # it at random init and defeat the point of a *learned* topology embedding.
        # (Previously this was `topo_embed.detach() if False else topo_embed`, which
        # is functionally identical to this but read as an accidental leftover.)
        l_topkd = F_.mse_loss(pooled, topo_embed)
        total = total + lam * l_topkd
        l_attn_value = float(l_topkd.detach().cpu())

    return total, ce, l_attn_value


## 8. Train / eval loops

In [ ]:

def make_optimizer(model):
    """Discriminative LR: the pretrained backbone gets CONFIG["LR_BACKBONE"] (gentler);
    every freshly-initialized module -- gate, residual projector, align head, topkd
    baseline -- gets the higher CONFIG["LR"]. Fixes the original single-LR-for-everything
    setup, which fine-tuned a pretrained CNN at a flat 1e-3 alongside randomly-init
    modules that actually need that rate to catch up (Section 1, item 8)."""
    backbone_params = list(model.backbone.parameters())
    new_params = []
    for mod in (model.gate, model.residual_projector, model.align_head, model.topkd_baseline):
        if mod is not None:
            new_params += list(mod.parameters())
    return torch.optim.Adam(
        [
            {"params": backbone_params, "lr": CONFIG["LR_BACKBONE"]},
            {"params": new_params, "lr": CONFIG["LR"]},
        ],
        lr=CONFIG["LR"],  # default lr for any param group added without one explicitly;
                          # unused now that all modules are materialized before optimizer
                          # construction (see the probe forward pass in train_one_run), but
                          # required by torch.optim.Adam's constructor regardless.
    )


def make_scheduler(optimizer, steps_per_epoch):
    """1-epoch linear warmup, then cosine decay to CONFIG["MIN_LR_FRAC"] of each param
    group's own base LR, applied once per optimizer step (not per epoch) -- the other half
    of the training-stability fix. A single shared lambda works correctly across the two
    differently-scaled param groups above, since LambdaLR multiplies each group's own
    initial_lr by lambda(step) rather than setting an absolute rate."""
    warmup_steps = CONFIG["WARMUP_EPOCHS"] * steps_per_epoch
    total_steps = CONFIG["EPOCHS"] * steps_per_epoch
    min_frac = CONFIG["MIN_LR_FRAC"]

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + float(np.cos(np.pi * progress)))
        return min_frac + (1.0 - min_frac) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


@torch.no_grad()
def evaluate(model, loader, use_topology, on_the_fly_topology=False):
    model.eval()
    model.use_topology = use_topology
    all_probs, all_labels = [], []

    for img, amap, entropy, labels, idx in loader:
        img, amap, entropy, labels = (img.to(DEVICE), amap.to(DEVICE),
                                       entropy.to(DEVICE).float(), labels.to(DEVICE))
        if on_the_fly_topology:
            # Oracle only: recompute BOTH A_gauss and entropy for this batch on the fly.
            # Reuses NB1's nuclei/PH pipeline -- see compute_oracle_maps() below.
            # BUGFIX: entropy must come from the same on-the-fly pass as amap -- the
            # dataset's own `entropy` for split!="train" is always 0.0 (placeholder),
            # which previously fed a wrong, constant value into the gate's gamma().
            amap, entropy = compute_oracle_maps(img)
            amap, entropy = amap.to(DEVICE), entropy.to(DEVICE).float()
        logits, _ = model(img, amap, entropy)
        probs = F_.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, all_probs > 0.5)
    return {"auc": auc, "acc": acc}


def train_one_run(cfg, seed):
    set_seed(seed)
    model = TopoDistilModel(
        backbone_name=CONFIG["BACKBONE"],
        gate_active=cfg["gate_active"], gate_form=cfg.get("gate_form", "A"),
        use_L_attn=cfg["use_L_attn"], use_residual=cfg["use_residual"],
        homology_mode=cfg["homology_mode"], baseline_type=cfg["baseline_type"],
        shuffled_control=cfg["shuffled_control"],
    ).to(DEVICE)

    train_ds = PCamTopoDataset(train_idx, "train", labels=train_labels, augment=True,
                                aug_mode="unaligned" if cfg["unaligned_aug"] else "aligned",
                                seed=seed)
    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                               num_workers=2, drop_last=True)

    val_ds = PCamTopoDataset(val_idx, "valid", labels=val_labels_sub, augment=False)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2)
    test_ds = PCamTopoDataset(test_idx, "test", labels=test_labels_sub, augment=False)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2)

    lam = 0.0 if cfg["name"] == "row5_gate_only" else CONFIG["LAMBDA"]

    # Materialize any lazily-created submodules (align_head, topkd_baseline -- their shape
    # depends on a real forward pass) BEFORE building the optimizer, so every trainable
    # parameter is present in the right param group from step 0 and gets the correct
    # warmup+cosine schedule from the very first update. Previously these were added via a
    # mid-training add_param_group() call, which LambdaLR does not retroactively cover --
    # a real (if minor) scheduling gap that this avoids entirely. eval() around the probe
    # so a throwaway batch doesn't perturb BatchNorm running stats.
    model.eval()
    model.use_topology = True
    with torch.no_grad():
        probe_img, probe_amap, probe_entropy, _, _ = next(iter(train_loader))
        model(probe_img.to(DEVICE), probe_amap.to(DEVICE), probe_entropy.to(DEVICE).float())
    model.train()

    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader))
    gate_means_per_epoch = []
    val_auc_per_epoch = []
    best_val_auc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        model.use_topology = True
        epoch_gate_means = []
        pbar = tqdm(train_loader, desc=f"{cfg['name']} seed{seed} epoch{epoch}", leave=False)

        for img, amap, entropy, labels, idx in pbar:
            img, amap, entropy, labels = (img.to(DEVICE), amap.to(DEVICE),
                                           entropy.to(DEVICE).float(), labels.to(DEVICE))
            logits, feat = model(img, amap, entropy)

            loss, ce, l_attn = compute_losses(model, logits, feat, img, amap, entropy,
                                               labels, lam, cfg)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            if model.gate is not None and model.gate.last_gate_mean is not None:
                epoch_gate_means.append(model.gate.last_gate_mean)
            pbar.set_postfix(loss=float(loss.detach().cpu()),
                              lr_head=optimizer.param_groups[1]["lr"])

        if epoch_gate_means:
            gate_means_per_epoch.append(float(np.mean(epoch_gate_means)))

        # Best-checkpoint tracking (Section 1, item 8): the original loop only ever
        # evaluated once, after the fixed 12th epoch, and reported whatever that happened
        # to be. Tracking val AUC every epoch and keeping the best-scoring state dict in
        # memory means the final reported numbers reflect the configuration's actual peak
        # performance on held-out data, not an arbitrary stopping point.
        epoch_val = evaluate(model, val_loader, use_topology=False)
        val_auc_per_epoch.append(epoch_val["auc"])
        if epoch_val["auc"] > best_val_auc:
            best_val_auc = epoch_val["auc"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    standard_val = evaluate(model, val_loader, use_topology=False)
    standard_test = evaluate(model, test_loader, use_topology=False)

    oracle_test = None
    if cfg["oracle_eval"]:
        oracle_loader = DataLoader(
            PCamTopoDataset(test_idx[:CONFIG["ORACLE_SUBSET"]], "test",
                            labels=test_labels_sub[:CONFIG["ORACLE_SUBSET"]], augment=False),
            batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2,
        )
        oracle_test = evaluate(model, oracle_loader, use_topology=True, on_the_fly_topology=True)

    return model, {
        "standard_val": standard_val,
        "standard_test": standard_test,
        "oracle_test": oracle_test,
        "gate_mean_per_epoch": gate_means_per_epoch,
        "val_auc_per_epoch": val_auc_per_epoch,   # new: per-epoch trajectory, for diagnosing
                                                    # whether training actually stabilized
        "best_epoch": best_epoch,                  # new: which epoch the reported model is from
    }


## 9. Oracle on-the-fly topology (test-time PH)

Only invoked for row 13 (and only on a small subset, `CONFIG["ORACLE_SUBSET"]`, since
this reintroduces the per-patch PH cost the whole method exists to remove). Reuses the
exact nuclei-extraction and persistent-homology functions from NB1 — copy them in here
rather than importing, since NB1 and NB2 are separate Kaggle sessions.


In [ ]:

!pip install -q ripser scikit-image
from skimage.color import rgb2hed
from skimage.filters import threshold_otsu
from skimage.measure import label as sklabel, regionprops
from skimage.morphology import remove_small_objects, binary_opening, disk
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform
from ripser import ripser as ripser_fn

# --- identical to NB1 Section 3/4/5 -- kept in sync manually; see NB1 for full docs.
def _extract_nuclei_centroids(patch_rgb, min_area=8):
    hed = rgb2hed(patch_rgb)
    h_channel = hed[:, :, 0]
    h_channel = (h_channel - h_channel.min()) / (np.ptp(h_channel) + 1e-8)
    try:
        thresh = threshold_otsu(h_channel)
    except ValueError:
        return np.zeros((0, 2))
    mask = h_channel > thresh
    mask = binary_opening(mask, footprint=disk(1))
    mask = remove_small_objects(mask, min_size=min_area)
    lbl = sklabel(mask)
    props = regionprops(lbl)
    return np.array([[p.centroid[1], p.centroid[0]] for p in props])

def _h0(centroids):
    n = len(centroids)
    if n < 2:
        return np.zeros(0), np.zeros((0, 2))
    dmat = squareform(pdist(centroids))
    mst = minimum_spanning_tree(csr_matrix(dmat)).tocoo()
    crit_xy = (centroids[mst.row] + centroids[mst.col]) / 2.0
    return mst.data, crit_xy

def _h1(centroids, dmat=None):
    n = len(centroids)
    if n < 3:
        return np.zeros(0), np.zeros((0, 2))
    dgm1 = ripser_fn(centroids, maxdim=1)["dgms"][1]
    dgm1 = dgm1[np.isfinite(dgm1[:, 1])]
    if len(dgm1) == 0:
        return np.zeros(0), np.zeros((0, 2))
    if dmat is None:
        dmat = squareform(pdist(centroids))
    iu = np.triu_indices(n, k=1)
    pair_dists = dmat[iu]
    crit_xy = np.zeros((len(dgm1), 2))
    for k, (b, d) in enumerate(dgm1):
        j = np.argmin(np.abs(pair_dists - b))
        p, q = iu[0][j], iu[1][j]
        crit_xy[k] = (centroids[p] + centroids[q]) / 2.0
    return dgm1[:, 1] - dgm1[:, 0], crit_xy

def _gauss_map(crit_xy, persistences, size, sigma=6.0):
    A = np.zeros((size, size), dtype=np.float32)
    if len(crit_xy) == 0:
        return A
    yy, xx = np.mgrid[0:size, 0:size]
    for (x, y), p in zip(crit_xy, persistences):
        if p <= 0:
            continue
        A += p * np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return A

def _persistence_entropy(persistences):
    '''H_i = -sum (p_i/P) log(p_i/P), P = sum p_i. Matches NB1's definition exactly.'''
    persistences = persistences[persistences > 0]
    if len(persistences) == 0:
        return 0.0
    P = persistences.sum()
    probs = persistences / P
    return float(-(probs * np.log(probs + 1e-12)).sum())


def compute_oracle_maps(img_batch):
    '''img_batch: (B,3,H,W) tensor in [0,1].
    Returns (amap, entropy): amap is (B,2,H,W), entropy is (B,2) = [entropy_h0, entropy_h1].

    BUGFIX: previously only the A_gauss map was recomputed on the fly for the oracle
    eval (row 13); entropy was left at the dataset's placeholder 0.0 (real entropy is
    only computed for split=="train"), so the gate's gamma() term used a constant,
    wrong entropy for every oracle sample instead of each patch's real value. Now both
    are computed together from the same on-the-fly persistent homology pass.
    '''
    imgs = (img_batch.permute(0, 2, 3, 1).cpu().numpy())
    size = CONFIG["IMG_SIZE"]
    out = np.zeros((len(imgs), 2, size, size), dtype=np.float32)
    entropy = np.zeros((len(imgs), 2), dtype=np.float32)
    for i, im in enumerate(imgs):
        centroids = _extract_nuclei_centroids(im)
        if len(centroids) < 3:
            continue
        p0, xy0 = _h0(centroids)
        p1, xy1 = _h1(centroids)
        out[i, 0] = _gauss_map(xy0, p0, size)
        out[i, 1] = _gauss_map(xy1, p1, size)
        entropy[i, 0] = _persistence_entropy(p0)
        entropy[i, 1] = _persistence_entropy(p1)
    return torch.from_numpy(out), torch.from_numpy(entropy)


## 10. Validation / test subsets

Full `valid`/`test` splits are large; per Section 4's feasibility budget we evaluate
standard metrics on a fixed random subset (`CONFIG["VAL_SUBSET"]`/`["TEST_SUBSET"]`) for
speed, held constant across all 25 runs so comparisons stay apples-to-apples.


In [ ]:

def load_labels(split, n):
    with h5py.File(pcam_files[f"{split}_y"], "r") as f:
        y_key = list(f.keys())[0]
        return np.array(f[y_key]).reshape(-1)[:n]

rng = np.random.default_rng(0)
n_valid_full = splits["n_valid"]
n_test_full = splits["n_test"]

y_valid_full = load_labels("valid", n_valid_full)
y_test_full = load_labels("test", n_test_full)

val_idx = rng.choice(n_valid_full, min(CONFIG["VAL_SUBSET"], n_valid_full), replace=False)
test_idx = rng.choice(n_test_full, min(CONFIG["TEST_SUBSET"], n_test_full), replace=False)
val_labels_sub = y_valid_full[val_idx]
test_labels_sub = y_test_full[test_idx]

print(f"Val subset: {len(val_idx)} | Test subset: {len(test_idx)}")


## 11. Experimental matrix (Section 4.3)

Translates the 13-row table into config dicts. Rows 1, 3, 4, 10, 11, 13 run at 3 seeds;
the rest at 1 seed — 25 runs total.

> **Update 1 (gate-free distillation-only follow-up):** row 4
> (`row4_lattn_only_no_gate`: `use_L_attn=True`, gate never applied, i.e. `F' = F`,
> distillation target is still the full residual-projected `A_proj`) was originally a
> single-seed mechanism-isolation row. Its 1-seed result (0.9358 test AUC, under the
> *old*, unscheduled training loop) beat full LIFE -- the leading candidate for "the
> gate isn't the part that's helping, the distillation loss might be." Promoted to
> `CORE_ROWS` for 3-seed, weight-saved treatment. (Under the old training loop this
> promotion already ran once and the effect did not survive 3 seeds -- see the run
> log. It is being kept as a core row under the *new*, stabilized training loop below,
> since the old result was itself confounded by the training instability this notebook
> now fixes.)
>
> **Update 2 (oracle promotion, fixes the n=1 oracle-ceiling issue):** row 13
> (`row13_oracle`) was a single-seed upper bound. Its one draw came out *below* both
> baseline and LIFE on RepViT -- an inversion that should be structurally impossible
> (the oracle has strictly more information at test time) and was flagged as
> inconclusive at n=1 rather than trusted. `CORE_ROWS` now includes row 13 too, so the
> oracle ceiling gets a real mean +/- std instead of one potentially-unlucky checkpoint,
> combined with the training-stability fix below (Section 1, item 8) which should also
> reduce how unlucky any single checkpoint can be in the first place.


In [ ]:

BASE = dict(gate_active=False, gate_form="A", use_L_attn=False, use_residual=True,
            homology_mode="both", baseline_type=None, shuffled_control=False,
            unaligned_aug=False, oracle_eval=False)

def cfg(name, **overrides):
    c = dict(BASE); c.update(overrides); c["name"] = name
    return c

EXPERIMENT_MATRIX = [
    cfg("row1_cnn_baseline", use_residual=False),
    cfg("row2_attention_transfer", use_L_attn=True, baseline_type="attention_transfer", use_residual=False),
    cfg("row3_topkd_style", baseline_type="topkd", use_residual=False),
    cfg("row4_lattn_only_no_gate", use_L_attn=True),
    cfg("row5_gate_only", gate_active=True, gate_form="A", use_L_attn=False),
    cfg("row6_fixed_gauss_only", gate_active=True, gate_form="A", use_L_attn=True, use_residual=False),
    cfg("row7_h0_only", gate_active=True, gate_form="A", use_L_attn=True, homology_mode="h0"),
    cfg("row8_h1_only", gate_active=True, gate_form="A", use_L_attn=True, homology_mode="h1"),
    cfg("row9_full_form_a", gate_active=True, gate_form="A", use_L_attn=True),
    cfg("row10_full_form_b", gate_active=True, gate_form="B", use_L_attn=True),
    cfg("row11_shuffled_control", gate_active=True, gate_form="B", use_L_attn=True, shuffled_control=True),
    cfg("row12_unaligned_aug", gate_active=True, gate_form="B", use_L_attn=True, unaligned_aug=True),
    cfg("row13_oracle", gate_active=True, gate_form="B", use_L_attn=True, oracle_eval=True),
]

CORE_ROWS = {"row1_cnn_baseline", "row3_topkd_style", "row4_lattn_only_no_gate",
             "row10_full_form_b", "row11_shuffled_control", "row13_oracle"}
# row4 promoted earlier; row13 (oracle) promoted now too -- see Section 11, Update 2

run_plan = []
for c in EXPERIMENT_MATRIX:
    seeds = CONFIG["SEEDS_CORE"] if c["name"] in CORE_ROWS else [CONFIG["SEED_SINGLE"]]
    for s in seeds:
        run_plan.append((c, s))

print(f"Total planned runs: {len(run_plan)}")
for c, s in run_plan:
    print(f"  {c['name']:28s} seed={s}")


## 12. Run the battery (resumable)

Each run is checked against `run_manifest.json` before starting — a killed/timed-out
session can simply re-run this cell and it picks up from the next incomplete run.


In [ ]:

manifest_path = f'{CONFIG["OUT_DIR"]}/run_manifest.json'
manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}

def run_id(cfg, seed):
    return f"{cfg['name']}__seed{seed}"

for c, seed in run_plan:
    rid = run_id(c, seed)
    if manifest.get(rid, {}).get("status") == "done":
        continue

    print(f"\n=== Running {rid} ===")
    t0 = time.time()
    try:
        model, metrics = train_one_run(c, seed)
    except Exception as e:
        print(f"FAILED {rid}: {e}")
        manifest[rid] = {"status": "failed", "error": str(e)}
        json.dump(manifest, open(manifest_path, "w"), indent=2)
        continue

    elapsed = time.time() - t0
    metrics["elapsed_sec"] = elapsed
    metrics["config"] = c
    metrics["seed"] = seed

    with open(f'{CONFIG["OUT_DIR"]}/{rid}_metrics.json', "w") as f:
        json.dump(metrics, f, indent=2)

    row_key = c["name"].split("_")[0]  # e.g. "row10"
    if row_key in CONFIG["SAVE_WEIGHTS_FOR"]:
        torch.save(model.state_dict(), f'{CONFIG["OUT_DIR"]}/{rid}_weights.pt')

    manifest[rid] = {"status": "done", "elapsed_sec": elapsed}
    json.dump(manifest, open(manifest_path, "w"), indent=2)

    del model
    torch.cuda.empty_cache()
    print(f"Done {rid} in {elapsed/60:.1f} min | "
          f"val AUC={metrics['standard_val']['auc']:.3f} "
          f"test AUC={metrics['standard_test']['auc']:.3f}")

print("\nBattery complete." if all(
    manifest.get(run_id(c, s), {}).get("status") == "done" for c, s in run_plan
) else "\nSome runs incomplete -- re-run this cell to resume.")


## 12b. Post-battery stability check (does the fix actually work?)

Reads back every `*_metrics.json` just written -- a round-trip check that the
checkpointing structure itself is sound -- and reports two things per core-credibility
config: (1) seed-mean +/- std test AUC, so you can compare directly against the *old*
loop's numbers (baseline was 0.9314 +/- 0.0268 the first time, 0.9217 +/- 0.0130 on the
from-scratch rerun); and (2) `best_epoch` per seed, which should *not* cluster at
`EPOCHS - 1` for every seed -- if it does, best-checkpoint selection isn't actually doing
anything different from the old "always take epoch 12" behavior, and the fix needs a
second look before trusting anything downstream.

In [ ]:
import glob

core_for_check = sorted(CORE_ROWS)
print(f"{'config':28s} {'n':3s} {'test_auc mean':14s} {'std':8s}   best_epoch per seed")
for cname in core_for_check:
    files = sorted(glob.glob(f'{CONFIG["OUT_DIR"]}/{cname}__seed*_metrics.json'))
    if not files:
        print(f"{cname:28s}  -- no metrics files found yet --")
        continue
    aucs, epochs = [], []
    for f in files:
        d = json.load(open(f))
        aucs.append(d["standard_test"]["auc"])
        epochs.append(d.get("best_epoch", -1))
    aucs = np.array(aucs)
    print(f"{cname:28s} {len(aucs):3d} {aucs.mean():.4f}         {aucs.std():.4f}   {epochs}")

print("\nIf every best_epoch above is EPOCHS-1 for every seed, the schedule/best-checkpoint")
print("fix isn\'t actually changing which checkpoint gets reported -- worth checking that")
print("val_auc_per_epoch (in each *_metrics.json) is genuinely non-monotonic before trusting")
print("the numbers above as evidence the training-instability fix worked.")


## 13. Model size / latency / FLOPs (Section 4.5)

In [ ]:

def profile_backbone_only(backbone_name=CONFIG["BACKBONE"], img_size=CONFIG["IMG_SIZE"]):
    '''Inference-time cost with the gate bypassed -- identical to the unmodified
    backbone, per Section 4.6 ("identical latency profile to the unmodified backbone").'''
    model = timm.create_model(backbone_name, pretrained=False, num_classes=2).to(DEVICE).eval()
    dummy = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    flops = None
    if HAS_THOP:
        flops, _ = thop_profile(model, inputs=(dummy,), verbose=False)

    with torch.no_grad():
        for _ in range(5):
            model(dummy)  # warmup
        torch.cuda.synchronize() if DEVICE == "cuda" else None
        t0 = time.time()
        for _ in range(50):
            model(dummy)
        torch.cuda.synchronize() if DEVICE == "cuda" else None
        latency_ms = (time.time() - t0) / 50 * 1000

    return {"params": n_params, "flops": flops, "latency_ms": latency_ms}

profile_result = profile_backbone_only()
with open(f'{CONFIG["OUT_DIR"]}/inference_profile.json', "w") as f:
    json.dump(profile_result, f, indent=2)
profile_result


## 14. Package checkpoint bundle for NB4 (and NB3's shared components)

In [ ]:

with open(f'{CONFIG["OUT_DIR"]}/config_used.json', "w") as f:
    json.dump({k: v for k, v in CONFIG.items() if k != "SAVE_WEIGHTS_FOR"} |
              {"SAVE_WEIGHTS_FOR": list(CONFIG["SAVE_WEIGHTS_FOR"])}, f, indent=2)

print("RepViT battery output:")
for f in sorted(Path(CONFIG["OUT_DIR"]).iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:10.2f} MB")

print()
print("Next: Save Version. In NB4, add this notebook's output as an input to load")
print("every *_metrics.json + run_manifest.json for the significance testing and")
print("results table. NB3 (MobileNetV4 cross-check) can reuse the model/gate classes")
print("and CONFIG assumptions defined here, but does not need this notebook's output")
print("as an input -- it only needs NB1's.")


## 15. Validation-only design search (injection layer x lambda)

Tests whether the fixed (layer, lambda) operating point used for the whole battery above
was already close to the best available, or whether a different point reveals a real
LIFE effect the current one is too weak to show. **Selection uses validation AUC only --
test AUC is not even printed until one winning configuration has been locked in.** This
is the honest version of "maybe the setup isn't apt enough": decide the configuration
from held-out validation performance, then measure it on test exactly once, rather than
trying configurations against test and keeping whichever looks best after the fact.

**Grid**: 3 injection depths (early/mid/late -- picked from `BatchNorm2d` layers only, so
every candidate is guaranteed a clean 4D feature map, the same kind of layer the original
injection point already was) x 3 lambda values (0.25 / 0.5 / 1.0) x gate form B (already
established as the primary form via rows 9 vs 10 in the main battery -- not re-opened
here, to keep the grid a manageable size). 9 configurations, **1 seed each** (seed 0,
matching every other single-seed mechanism-isolation row in the main battery -- this
stage is a hyperparameter search, not a claim, so it doesn't need 3 seeds any more than
rows 2/5-9/12 did).

**The winning configuration then gets exactly one confirmatory run at the same
`SEEDS_CORE = [0, 1, 2]` used for every other core row -- not more, not fewer** -- so
it's directly comparable to row 10 and row 1, and doesn't get more chances at a good
result than anything else in the paper.

In [ ]:
# --- Build injection-layer candidates: BatchNorm2d modules only, so every candidate is
# guaranteed a clean 4D (B, C, H, W) output (matches how the original injection point --
# a *.bn layer -- was chosen in Section 3).
_probe = timm.create_model(CONFIG["BACKBONE"], pretrained=True, num_classes=2)
bn_names = [n for n, m in _probe.named_modules() if isinstance(m, nn.BatchNorm2d)]
del _probe

def _pick_depth(frac):
    return bn_names[int(round(frac * (len(bn_names) - 1)))]

LAYER_CANDIDATES = {
    "early": _pick_depth(0.25),
    "mid":   CONFIG["INJECTION_LAYER"],   # the layer the whole battery above already used
    "late":  _pick_depth(0.75),
}
LAMBDA_CANDIDATES = [0.25, 0.5, 1.0]

print("Injection-layer candidates for the search:")
for k, v in LAYER_CANDIDATES.items():
    print(f"  {k:6s}: {v}")
print(f"Lambda candidates: {LAMBDA_CANDIDATES}")

SEARCH_GRID = [
    {"layer_key": lk, "layer": lv, "lambda": lam}
    for lk, lv in LAYER_CANDIDATES.items()
    for lam in LAMBDA_CANDIDATES
]
print(f"\n{len(SEARCH_GRID)} search configurations, 1 seed each, selected on validation AUC only.")


In [ ]:
search_manifest_path = f'{CONFIG["OUT_DIR"]}/design_search_manifest.json'
search_manifest = json.load(open(search_manifest_path)) if os.path.exists(search_manifest_path) else {}

_orig_injection_layer = CONFIG["INJECTION_LAYER"]
_orig_lambda = CONFIG["LAMBDA"]
search_results = []

for point in SEARCH_GRID:
    key = f"search__{point['layer_key']}__lam{point['lambda']}"
    if search_manifest.get(key, {}).get("status") == "done":
        print(f"[skip] {key} already done")
        search_results.append(search_manifest[key])
        continue

    CONFIG["INJECTION_LAYER"] = point["layer"]
    CONFIG["LAMBDA"] = point["lambda"]
    search_cfg = cfg(key, gate_active=True, gate_form="B", use_L_attn=True)  # same flags as row10

    print(f"=== Running {key} (layer={point['layer']}, lambda={point['lambda']}) ===")
    t0 = time.time()
    model, metrics = train_one_run(search_cfg, seed=CONFIG["SEED_SINGLE"])
    elapsed = time.time() - t0
    print(f"Done in {elapsed/60:.1f} min | val AUC={metrics['standard_val']['auc']:.4f} "
          f"(test AUC computed but withheld from selection)")

    entry = {
        "layer_key": point["layer_key"], "layer": point["layer"], "lambda": point["lambda"],
        "val_auc": metrics["standard_val"]["auc"],
        "test_auc_DO_NOT_USE_FOR_SELECTION": metrics["standard_test"]["auc"],
        "elapsed_sec": elapsed, "status": "done",
    }
    search_manifest[key] = entry
    json.dump(search_manifest, open(search_manifest_path, "w"), indent=2)
    search_results.append(entry)
    del model
    torch.cuda.empty_cache()

CONFIG["INJECTION_LAYER"] = _orig_injection_layer
CONFIG["LAMBDA"] = _orig_lambda
print("\nSearch complete. Restored CONFIG to the original battery's (layer, lambda).")


In [ ]:
# Select the winner by validation AUC ONLY. Test AUC for every search point is shown
# here purely for transparency/record-keeping -- it must not (and did not) influence
# which configuration was picked.
best = max(search_results, key=lambda e: e["val_auc"])
print("All search points, sorted by validation AUC:")
for e in sorted(search_results, key=lambda e: -e["val_auc"]):
    marker = "  <-- selected" if e is best else ""
    print(f"  {e['layer_key']:6s} lambda={e['lambda']:<5} val_auc={e['val_auc']:.4f}  "
          f"(test_auc={e['test_auc_DO_NOT_USE_FOR_SELECTION']:.4f}, for the record only){marker}")

print(f"\nSelected: layer={best['layer']} ({best['layer_key']}), lambda={best['lambda']}, "
      f"val_auc={best['val_auc']:.4f}")

CONFIG["INJECTION_LAYER"] = best["layer"]
CONFIG["LAMBDA"] = best["lambda"]
confirm_cfg = cfg("row14_design_search_winner", gate_active=True, gate_form="B", use_L_attn=True)
CONFIG["SAVE_WEIGHTS_FOR"].add("row14")   # so it's checkpointed the same way as every core row

for seed in CONFIG["SEEDS_CORE"]:
    run_key = f"row14_design_search_winner__seed{seed}"
    if manifest.get(run_key, {}).get("status") == "done":
        print(f"[skip] {run_key} already done")
        continue
    print(f"=== Running {run_key} (confirmatory, validation-selected config) ===")
    t0 = time.time()
    model, metrics = train_one_run(confirm_cfg, seed=seed)
    elapsed = time.time() - t0
    json.dump(metrics, open(f'{CONFIG["OUT_DIR"]}/{run_key}_metrics.json', "w"), indent=2,
               default=lambda o: o.tolist() if hasattr(o, "tolist") else o)
    torch.save(model.state_dict(), f'{CONFIG["OUT_DIR"]}/{run_key}_weights.pt')
    manifest[run_key] = {"status": "done", "elapsed_sec": elapsed}
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    print(f"Done in {elapsed/60:.1f} min | val AUC={metrics['standard_val']['auc']:.4f} "
          f"test AUC={metrics['standard_test']['auc']:.4f}")
    del model
    torch.cuda.empty_cache()

CONFIG["INJECTION_LAYER"] = _orig_injection_layer
CONFIG["LAMBDA"] = _orig_lambda
print("\nRestored CONFIG to the original battery's (layer, lambda) for anything run after this.")


In [ ]:
# Honest comparison: row14 (validation-selected) vs row10 (original fixed operating
# point) vs row1 (baseline), same SEEDS_CORE for all three -- report whatever this says,
# in either direction. This is a *labeled*, post-hoc-selected confirmatory row, reported
# separately from the pre-registered 13-row battery above, not folded into it.
def _row_test_aucs(cfg_name):
    return [json.load(open(f'{CONFIG["OUT_DIR"]}/{cfg_name}__seed{s}_metrics.json'))["standard_test"]["auc"]
            for s in CONFIG["SEEDS_CORE"]]

b = np.array(_row_test_aucs("row1_cnn_baseline"))
l = np.array(_row_test_aucs("row10_full_form_b"))
w = np.array(_row_test_aucs("row14_design_search_winner"))

print(f"{'':30s} {'seed0':>8s} {'seed1':>8s} {'seed2':>8s} {'mean':>8s} {'std':>8s}")
for name, arr in [("row1 baseline", b), ("row10 LIFE (fixed op. point)", l),
                   ("row14 LIFE (validation-selected)", w)]:
    print(f"{name:30s} " + " ".join(f"{v:8.4f}" for v in arr) + f" {arr.mean():8.4f} {arr.std():8.4f}")

print(f"\nrow14 - row1  (does the searched config beat baseline?): {(w-b).round(4)}  mean={np.mean(w-b):+.4f}")
print(f"row14 - row10 (does the searched config beat the original fixed point?): "
      f"{(w-l).round(4)}  mean={np.mean(w-l):+.4f}")
print("\nReport whichever of these is true -- a validation-selected config that still")
print("doesn't beat baseline is real evidence the fixed operating point wasn't the")
print("problem, exactly as informative for the paper as a config that does.")


## 16. Subgroup analysis: does LIFE help more on high-topology-entropy patches? (zero retraining)

**Pre-registered hypothesis, stated before looking at any result below**: LIFE's entropy
gate `gamma = sigmoid(sum(H_i) - H_THRESH)` (Section 3, Eq. for gamma) makes the topology
gate close to a no-op (`gamma ~ 0`, so `F' ~ F`) on patches whose summed persistence
entropy falls *below* `CONFIG["H_THRESH"]` -- the median over the **training** subsample,
reused here completely unchanged, not re-tuned on test data. If LIFE has a real effect
that whole-test-set-average AUC is diluting by mixing it with patches where the gate is
mechanically almost off anyway, it should show up as a larger baseline -> LIFE delta on
high-entropy test patches than on low-entropy ones.

Every `standard_test` number reported elsewhere in this notebook evaluates the deployed
model with the gate fully bypassed (`use_topology=False`, Section 1 assumption 1) -- so
this specifically asks whether *training-time* exposure to topology left a differential
benefit on patches that were topologically complex, even though the gate never runs at
test time. That is a different question from row 13's oracle (which asks what happens if
the gate is switched back on at test time too).

Uses the row 1 (baseline) and row 10 (full LIFE) checkpoints **already saved** by the
main battery above -- zero retraining, one forward pass per model per test patch.

In [ ]:
# Recompute topology entropy for the *test* set on the fly, once. This is a property of
# the data only (independent of which model/seed is evaluated against it), computed a
# single time and reused below for every model. The dataset's own `entropy` field is a
# placeholder (0.0) for split != "train" (see compute_oracle_maps's docstring) -- real
# per-patch entropy for test patches only exists by recomputing it here, the same way
# row 13's oracle eval already does.
test_loader_full = DataLoader(
    PCamTopoDataset(test_idx, "test", labels=test_labels_sub, augment=False),
    batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2,
)

test_entropy_sum, test_labels_ordered = [], []
with torch.no_grad():
    for img, amap, entropy, labels, idx in tqdm(test_loader_full, desc="Computing test-set entropy (one-time)"):
        _, ent = compute_oracle_maps(img)
        test_entropy_sum.append(ent.sum(axis=1))
        test_labels_ordered.append(labels.numpy())
test_entropy_sum = np.concatenate(test_entropy_sum)
test_labels_ordered = np.concatenate(test_labels_ordered)

high_mask = test_entropy_sum >= CONFIG["H_THRESH"]
print(f"High-entropy test patches: {high_mask.sum()} / {len(high_mask)} ({high_mask.mean()*100:.1f}%), "
      f"split at the pre-specified, train-derived threshold H_THRESH={CONFIG['H_THRESH']:.3f} "
      f"(not re-tuned on test data).")


In [ ]:
def load_saved_model(cfg_name, seed, cfg_dict):
    weights_path = f'{CONFIG["OUT_DIR"]}/{cfg_name}__seed{seed}_weights.pt'
    model = TopoDistilModel(
        backbone_name=CONFIG["BACKBONE"], gate_active=cfg_dict["gate_active"],
        gate_form=cfg_dict.get("gate_form", "A"), use_L_attn=cfg_dict["use_L_attn"],
        use_residual=cfg_dict["use_residual"], homology_mode=cfg_dict["homology_mode"],
        baseline_type=cfg_dict["baseline_type"], shuffled_control=cfg_dict["shuffled_control"],
    ).to(DEVICE)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.eval()
    return model

@torch.no_grad()
def get_test_predictions(model):
    model.use_topology = False   # deployed, inference-free configuration -- gate bypassed,
                                  # identical to every standard_test number elsewhere (Section 1)
    probs = []
    for img, amap, entropy, labels, idx in test_loader_full:
        logits, _ = model(img.to(DEVICE), amap.to(DEVICE), entropy.to(DEVICE).float())
        probs.append(F_.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

row1_cfg = next(c for c in EXPERIMENT_MATRIX if c["name"] == "row1_cnn_baseline")
row10_cfg = next(c for c in EXPERIMENT_MATRIX if c["name"] == "row10_full_form_b")

subgroup_rows = []
for seed in CONFIG["SEEDS_CORE"]:
    base_model = load_saved_model("row1_cnn_baseline", seed, row1_cfg)
    life_model = load_saved_model("row10_full_form_b", seed, row10_cfg)
    base_probs = get_test_predictions(base_model)
    life_probs = get_test_predictions(life_model)
    del base_model, life_model
    torch.cuda.empty_cache()

    for group_name, mask in [("high_entropy", high_mask), ("low_entropy", ~high_mask)]:
        base_auc = roc_auc_score(test_labels_ordered[mask], base_probs[mask])
        life_auc = roc_auc_score(test_labels_ordered[mask], life_probs[mask])
        subgroup_rows.append({"seed": seed, "group": group_name, "n": int(mask.sum()),
                               "baseline_auc": base_auc, "life_auc": life_auc,
                               "delta": life_auc - base_auc})
        print(f"seed{seed} {group_name:12s} (n={mask.sum():4d}): "
              f"baseline={base_auc:.4f}  LIFE={life_auc:.4f}  delta={life_auc - base_auc:+.4f}")

subgroup_df = pd.DataFrame(subgroup_rows)
out_path = f'{CONFIG["OUT_DIR"]}/subgroup_entropy_analysis.csv'
subgroup_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

print("\nMean delta by subgroup (LIFE - baseline), across the same 3 seeds used everywhere else:")
print(subgroup_df.groupby("group")["delta"].agg(["mean", "std"]))
print("\nIf the high_entropy delta is meaningfully more positive than low_entropy's delta,")
print("that is evidence for the pre-registered hypothesis above. If both are similarly flat")
print("(or both negative, consistent with Section 12's whole-test-set finding), that is")
print("evidence the gate genuinely isn't helping even where it's most active -- also a real,")
print("reportable answer.")
